# Data Cleaning Notebook

## Overview

In this notebook, I:
	•	Load the raw dataset from data/raw/.
	•	Inspect & Filter the data to keep tweets about Apple and Google only.
	•	Map the original sentiment labels to three categories: positive, negative, and neutral.
	•	Clean the tweet text by removing URLs, Twitter handles, punctuation, and extra whitespace.
	•	Save the cleaned dataset in data/processed/ for further analysis.
	

---

## 1. Load & Inspect the Raw Data

I load the CSV file containing over 9,000 tweets and check the data’s shape and structure to understand how many rows and columns I have.

In [2]:
import pandas as pd
import re
import numpy as np

import nltk   

pd.set_option('display.max_colwidth', 200)

In [4]:
df = pd.read_csv('../Twitter-Sentiment-Analysis/data/raw/brands-and-product-emotions.csv', encoding='latin1')

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [5]:
df['is_there_an_emotion_directed_at_a_brand_or_product'].value_counts()

is_there_an_emotion_directed_at_a_brand_or_product
No emotion toward brand or product    5389
Positive emotion                      2978
Negative emotion                       570
I can't tell                           156
Name: count, dtype: int64

---

## 2. Map & Filter Sentiment

Originally, the column `is_there_an_emotion_directed_at_a_brand_or_product` contained four main categories:
1. `Positive emotion`
2. `Negative emotion`
3. `No emotion toward brand or product`
4. `I can't tell`

- **Step**: I create a `sentiment_map` dict to convert these categories into `positive`, `negative`, or `neutral`.
- **Reason**: My goal is to build a model that classifies tweets into **positive**, **negative**, or **neutral**.
- **Action**: I drop `"I can't tell"` rows since they're ambiguous.

In [6]:
# Creating a dictionary to map the existing labels to simpler sentiment labels
sentiment_map = {
    "Positive emotion": "positive",
    "Negative emotion": "negative",
    "No emotion toward brand or product": "neutral"
}

# Filtering out "I can't tell" rows
df = df[df['is_there_an_emotion_directed_at_a_brand_or_product'] != "I can't tell"]

# Now mapping the remaining rows to 'positive', 'negative', or 'neutral'
df['sentiment'] = df['is_there_an_emotion_directed_at_a_brand_or_product'].map(sentiment_map)
df['sentiment'].value_counts()

sentiment
neutral     5389
positive    2978
negative     570
Name: count, dtype: int64

---

## 3. Filter for Apple & Google

My project focuses on **Apple** and **Google** products only. The column `emotion_in_tweet_is_directed_at` indicates which brand the tweet references.

- **Step**: Filter rows where `emotion_in_tweet_is_directed_at` is either `"Apple"` or `"Google"`.
- **Reason**: Narrow the dataset to tweets about these two specific brands.
- **Outcome**: This significantly reduces the dataset size but ensures brand relevance.

In [7]:
desired_brands = ["Apple", "Google"]
df = df[df['emotion_in_tweet_is_directed_at'].isin(desired_brands)]

In [8]:
df.shape
df['emotion_in_tweet_is_directed_at'].value_counts()
df['sentiment'].value_counts()

sentiment
positive    889
negative    163
neutral      36
Name: count, dtype: int64

---

## 4. Drop Missing or Irrelevant Rows

To avoid errors and improve data quality:

- **Step**: Drop rows where `tweet_text` is null (or empty).
- **Reason**: I cannot analyze a tweet if it doesn’t exist or is blank.

In [9]:
df = df.dropna(subset=['tweet_text'])

In [10]:
df.shape

(1088, 4)

---

## 5. Clean Tweet Text

I apply a function `clean_tweet(text)` that:
1. Converts text to lowercase.
2. Removes URLs (e.g., `http://...`).
3. Removes Twitter handles (words starting with `@`).
4. Removes punctuation.
5. Strips extra whitespace.

- **Step**: `df['cleaned_text'] = df['tweet_text'].apply(clean_tweet)`
- **Reason**: Standardize textual data so that it’s easier for vectorizers and NLP models to process.
- **Example**: A tweet `"@Apple I'm loving this new iPhone! http://bit.ly/..."` becomes `"im loving this new iphone"`.

In [11]:
def clean_tweet(text):
    text = text.lower()
    # remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # remove Twitter handles
    text = re.sub(r'@\w+', '', text)
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the function
df['cleaned_text'] = df['tweet_text'].apply(clean_tweet)
df[['tweet_text','cleaned_text']].head()

,tweet_text,cleaned_text
4,"@sxtxstate great stuff on Fri #SXSW: Marissa Mayer (Google), Tim O'Reilly (tech books/conferences) &amp; Matt Mullenweg (Wordpress)",great stuff on fri sxsw marissa mayer google tim oreilly tech booksconferences amp matt mullenweg wordpress
9,Counting down the days to #sxsw plus strong Canadian dollar means stock up on Apple gear,counting down the days to sxsw plus strong canadian dollar means stock up on apple gear
38,@mention - False Alarm: Google Circles Not Coming NowÛÒand Probably Not Ever? - {link} #Google #Circles #Social #SXSW,false alarm google circles not coming nowûòand probably not ever link google circles social sxsw
40,@mention - Great weather to greet you for #sxsw! Still need a sweater at night..Apple putting up &quot;flash store&quot; downtown to sell iPad2,great weather to greet you for sxsw still need a sweater at nightapple putting up quotflash storequot downtown to sell ipad2
47,HOORAY RT ÛÏ@mention Apple Is Opening A Pop-Up Store In Austin For #SXSW | @mention {link},hooray rt ûï apple is opening a popup store in austin for sxsw link


---

## 6. Handle Duplicates

If my dataset has repeated tweets or retweets:

- **Step**: Remove duplicates based on `cleaned_text`.
- **Reason**: Duplicate rows can bias the model if certain tweets appear multiple times.

In [12]:
initial_size = df.shape[0]
df.drop_duplicates(subset=['cleaned_text'], inplace=True)
print("Removed", initial_size - df.shape[0], "duplicates.")

Removed 16 duplicates.


---

## 7. Final Shape & Save

After cleaning, I display the final shape of my dataframe (`df.shape`) and confirm the class distribution for `sentiment` (`df['sentiment'].value_counts()`).

- **Step**: `df.to_csv('data/processed/cleaned_tweets.csv', index=False)`
- **Reason**: So future notebooks (EDA, modeling) can access a standardized dataset without repeating all cleaning steps.

In [15]:
df.shape

(1072, 5)

In [13]:
df.to_csv('data/processed/cleaned_tweets.csv', index=False)

---

## Conclusion

This completes my **data cleaning** process. Next, I will move on to exploring and visualizing this dataset in the `02_eda.ipynb` notebook, where I will analyze the distribution of sentiments, word usage, and more.